# Tools (도구)

OpenAI 플랫폼에서 사용할 수 있는 도구에 대한 개요는 다음과 같습니다.

| 도구 | 설명 |
|---|---|
| Function calling | 커스텀 코드를 호출하여 모델에 추가 데이터 및 기능 제공 |
| Web search | 인터넷 데이터를 모델 응답 생성에 포함 |
| Remote MCP servers | MCP 서버를 통해 모델에 새로운 기능 추가 |
| Skills | 버전 관리된 스킬 번들을 호스팅 환경에 업로드하여 재사용 |
| Shell | 호스팅 컨테이너 또는 로컬 런타임에서 쉘 명령 실행 |
| Computer use | 모델이 컴퓨터 인터페이스를 제어하는 에이전틱 워크플로우 생성 |
| Image generation | GPT Image를 사용하여 이미지 생성 또는 편집 |
| File search | 업로드된 파일 내용을 검색하여 응답 생성 시 컨텍스트로 활용 |
| Tool search | 관련 도구를 동적으로 로드하여 토큰 사용량 및 지연시간 최적화 |

### 내장 도구 비용

* 참고 : [Built-in tools Pricing](https://openai.com/api/pricing/)

### 웹검색 (Web Search)
모델이 응답을 생성하기 전에 최신 정보를 웹에서 검색할 수 있도록 허용

- 사용자 위치 지정

### File Search

In [ ]:
# "Financial Statements"라는 벡터 스토어 생성
# OpenAI에 업로드할 파일 준비
# 업로드 및 폴링 SDK 도우미를 사용하여 파일을 업로드하고 벡터 스토어에 추가,
# 파일 배치의 완료 상태를 폴링합니다.
# 이 작업의 결과를 보기 위해 상태 및 파일 개수를 출력할 수 있습니다.

### Remote MCP
- 모델이 원격 MCP 서버를 사용하여 작업을 수행할 수 있도록 허용
- 모델 컨텍스트 프로토콜 (MCP)은 애플리케이션이 LLM에 도구와 컨텍스트를 제공하는 방식을 표준화
- Responses API의 MCP 도구를 통해 개발자는 모델에 원격 MCP 서버 에 호스팅된 도구에 대한 액세스 권한을 부여
- 원격 MCP 서버는 개발자와 조직이 인터넷을 통해 관리하며, Responses API와 같은 도구를 MCP 클라이언트에 제공하는 MCP 서버

In [ ]:
#  DeepWiki MCP 서버를 사용하여 거의 모든 공개 저장소에 대해 질문

---------------

### 여러 도구들을 활용한 챗봇 구축
- 여러 도구(웹검색, 파일검색, MCP)를 **하나의 요청에 함께 등록**하여 사용하는 방법을 보여줍니다.
- 대화 히스토리를 **Python `list`로 직접 관리**합니다 (클라이언트 측 상태 관리).

> **참고 — `230_Conversation_state.ipynb`와의 차이**
> - **이 노트북**: `conversation_history` 리스트를 직접 누적하여 매 요청마다 전체 대화를 `input`으로 전달 → **수동 / 클라이언트 측** 상태 관리
> - **230 노트북**: `previous_response_id`를 사용하여 OpenAI 서버가 상태를 자동 보존 → **서버 측** 상태 관리

---------------
### Skills
- **버전 관리된 스킬 번들**(`SKILL.md` + 보조 파일들의 폴더)을 OpenAI에 **업로드**하여 재사용
- 스킬은 독립 도구가 아니라 **`shell` 도구의 실행 환경(`environment.skills`)에 부착**하여 사용
- 동작 흐름: ① 스킬 폴더 업로드 → `skill_id` 획득 → ② shell 도구에 부착 → ③ 모델이 컨텍스트에서 스킬 이름·설명을 보고 필요 시 `SKILL.md`를 읽어 실행

**(1) 스킬 번들 구조 예시**
```
my_skill/
├── SKILL.md        # 메타데이터(name, description) + 모델용 지침
└── script.py       # 보조 스크립트, API 명세, 리소스 등
```

**(2) 스킬 업로드** — `POST https://api.openai.com/v1/skills`
```python
import requests, os

# 방법 A) 폴더 내 파일들을 멀티파트로 업로드
resp = requests.post(
    "https://api.openai.com/v1/skills",
    headers={"Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}"},
    files=[
        ("files[]", ("SKILL.md", open("my_skill/SKILL.md", "rb"))),
        ("files[]", ("script.py", open("my_skill/script.py", "rb"))),
    ],
)
skill_id = resp.json()["id"]
print(skill_id)

# 방법 B) 단일 최상위 폴더를 압축한 zip 업로드
# requests.post(url, headers=..., files={"files": ("skill.zip", open("skill.zip","rb"), "application/zip")})
```

**(3) Responses API에서 사용** — `shell` 도구의 `environment.skills`에 부착
```python
response = client.responses.create(
    model=Model,
    tools=[{
        "type": "shell",
        "environment": {
            "type": "container_auto",
            "skills": [
                {"type": "skill_reference", "skill_id": skill_id},
                # 특정 버전 고정도 가능: {"type": "skill_reference", "skill_id": skill_id, "version": 2}
            ],
        },
    }],
    input="업로드한 스킬을 사용하여 작업을 수행해 주세요."
)

print(response.output_text)
```

> 참고: Skills는 `shell` 도구(호스팅 컨테이너 실행) 위에서 동작하므로, 이를 지원하는 모델과 계정 권한이 필요합니다. 위 코드는 개념 설명용 예시입니다.

#### 실습 예제: "투자 수익률(ROI) 분석 스킬"

스킬의 개념을 직관적으로 이해하기 위해, 간단하지만 실용적인 스킬을 직접 업로드해 사용해 봅니다.

**왜 스킬인가?** — 다음 두 가지를 한 번 패키지로 묶어 두면, 매번 긴 지침을 반복해서 붙여넣을 필요가 없습니다.
1. **표준 리포트 형식** (`SKILL.md`의 지침) → 매번 같은 양식으로 결과 정리
2. **정확한 계산** (`roi_calculator.py` 스크립트) → LLM이 약한 수치 계산을 스크립트가 대신 처리

**스킬 번들 구성** (`skills/roi_report/` 폴더):
```
skills/roi_report/
├── SKILL.md           # 스킬 이름·설명 + 출력 형식 + "계산은 스크립트로" 지침
└── roi_calculator.py  # 원금/현재가/기간 → 총수익률, 연환산수익률(CAGR) 계산
```

**동작 흐름**: 스킬 업로드(`skill_id` 획득) → `shell` 도구에 부착 → 사용자가 투자 정보를 물으면, 모델이 스킬 설명을 보고 `roi_calculator.py`를 실행한 뒤 표준 리포트 형식으로 답변.

In [ ]:
# (1) 스킬 번들 업로드 : skills/roi_report 폴더의 파일들을 멀티파트로 전송
#     성공하면 skill_id 를 돌려받아 이후 요청에서 재사용한다.
#     주의) 모든 파일은 "하나의 최상위 폴더" 아래에 있어야 하므로
#           파일명 앞에 공통 폴더명(roi_report/)을 붙여서 전송한다.

In [ ]:
# (2) 업로드한 스킬을 shell 도구에 부착하여 사용
#     모델이 질문을 보고 스스로 roi_calculator.py 를 실행한 뒤 표준 리포트로 답변한다.
#
#     주의) shell 도구의 environment(컨테이너 + Skills)는 gpt-5.2 이상에서만 지원된다.
#           (gpt-5-nano, gpt-5.1 등에서는 사용 불가)

---------------
### Shell
- 호스팅 컨테이너 또는 로컬 런타임에서 쉘(shell) 명령을 실행
- 모델이 파일 조작, 스크립트 실행, 시스템 명령 수행 등 터미널 작업을 직접 처리
- 코드 실행이나 환경 조작이 필요한 에이전틱 작업에 활용

> 주의: shell 도구와 컨테이너 실행 환경(`environment`)은 **gpt-5.2 이상**에서만 동작합니다. 아래 셀은 이를 지원하는 모델로 실제 실행해 보는 예제입니다.

In [ ]:
# shell 도구로 컨테이너 안에서 쉘 명령을 실행시키는 예제
#   - environment(container_auto)를 주면 모델이 격리된 컨테이너에서 명령을 실행한다.
#   - gpt-5.2 이상 필요 (gpt-5-nano, gpt-5.1 미지원)
#
#   포인트) "1+...+100" 같은 질문은 모델이 암산으로도 답할 수 있어 shell 사용을 증명하지 못한다.
#           그래서 "실제 컨테이너에서 명령을 실행해야만 알 수 있는 것"(OS/파이썬 버전 등)을 물어본다.
# 모델이 실제로 shell 명령을 실행했는지 출력 항목의 타입으로 확인
#   (shell 관련 항목이 보이면 컨테이너에서 명령이 실제 실행된 것)

---------------
### Tool search
- 사용 가능한 도구가 많을 때, 관련 도구를 동적으로 로드하여 토큰 사용량 및 지연시간(latency)을 최적화
- 모든 도구 정의를 매 요청마다 모델에 전달하는 대신, 요청과 관련된 도구만 선별하여 컨텍스트에 포함
- 대규모 도구 집합(특히 여러 MCP 서버 연결)을 다룰 때 효율성을 크게 향상

**사용법**
- `tools`에 `{"type": "tool_search"}`를 추가하고, 지연 로드할 도구(MCP 서버 등)에는 `defer_loading: True`를 지정
- `tool_search`는 **gpt-5.4 이상**에서만 동작

**효과 확인 방법**
- 같은 질문을 ① 모든 도구 선(先)로드 vs ② `tool_search` 지연 로드로 보내고 **입력 토큰 수(`response.usage.input_tokens`)를 비교**
- `response.output`에 `tool_search_call` 항목이 나타나면 모델이 도구를 동적으로 검색·로드한 것

> 참고: 토큰 절감 효과는 **등록한 도구(함수) 수가 많을수록 커집니다.** 아래 예제는 DeepWiki MCP(도구 3개)라 절감폭이 작을 수 있지만, 동작 원리와 측정 방법을 확인하는 데 목적이 있습니다.

In [ ]:
# Tool search 효과 비교 : 같은 질문을 (A) 모든 도구 선로드 vs (B) tool_search 지연로드 로 보내고
#                          입력 토큰 수를 비교한다. (tool_search 는 gpt-5.4 이상 필요)
# tool_search 방식 : defer_loading=True 로 도구를 지연 로드하고, 필요할 때만 검색해서 로드
# tool_search 가 실제로 작동했는지 출력 항목 타입으로 확인 --> tool_search_call 이 보이면 동적 로드 작동

### 실습 문제

웹 검색 도구(`web_search`)를 사용하여 **실시간 정보**를 묻는 요청을 작성하세요.

1. `tools=[{"type": "web_search"}]` 를 사용하세요.
2. 오늘 시점의 정보가 필요한 질문(예: 환율, 날씨, 뉴스)을 하나 던지세요.
3. `response.output_text` 로 답변을 출력하세요.

### 테스트 입력 예시

- `"오늘 원/달러 환율은 얼마인가요?"`
- `"이번 주 서울 날씨 전망을 알려줘."`